# Self-Consistency CoT Prototype & Classical Benchmarking
This notebook implements the Demonstration Plan mirroring `docs/demo_plan.md`.
We will compare three different execution paths:
1. **Classical Baseline**: Running a traditional fine-tuned Seq2Seq model (flan-t5-small) via HuggingFace.
2. **Greedy Generation (N=1)**: Prompting Gemini-2.5-Flash deterministically.
3. **Self-Consistency (N=5)**: Polling parallel Gemini instances using a high temperature and algebraically capturing the mode.


In [1]:
# In case dependencies are missing in the local environment, install them directly here.
!pip install -q transformers torch langchain langchain-google-genai python-dotenv nest_asyncio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.8 MB/s eta 0:00:00


In [2]:
import re
import asyncio
import os
import getpass
import nest_asyncio
from collections import Counter
from IPython.display import display, HTML
from dotenv import load_dotenv

# Allow nested event loops in Jupyter
nest_asyncio.apply()

load_dotenv()

# Securely prompt the user for an API key if it isn't set via .env
if "GEMINI_API_KEY" not in os.environ:
    print("Google Gemini API key not found in environment.")
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Please enter your GEMINI_API_KEY: ")

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage


Google Gemini API key not found in environment.
Please enter your GEMINI_API_KEY: ··········


In [48]:
async def gemini_generate(prompt: str, temperature: float) -> str:
    # Use gemini-1.5-flash for stable testing
    try:
        model = ChatGoogleGenerativeAI(
            model="gemini-2.5-flash-lite",
            temperature=temperature,
            max_output_tokens=2048
        )

        system_instructions = (
            "You are an expert logical reasoning agent. "
            "You must think step-by-step through the user's problem. "
            "CRITICAL: At the very end of your response, you MUST provide your final distinct conclusion formatted EXACTLY as: 'Final Answer: [Your Exact Answer]'"
        )

        messages = [
            SystemMessage(content=system_instructions),
            HumanMessage(content=prompt)
        ]

        response = await model.ainvoke(messages)
        # Ensure we return the content as a string
        content = response.content
        if isinstance(content, list):
            return " ".join([str(item.get('text', '')) if isinstance(item, dict) else str(item) for item in content])
        return str(content)
    except Exception as e:
        return f"Error: {str(e)}"

In [49]:
def extract_answer(text: str) -> str:
    # Improved regex to be more flexible with formatting
    match = re.search(r'final answer:\s*\[?(.*?)\]?(?=\n|$)', text, re.IGNORECASE)
    if match:
        answer = match.group(1).strip()
        answer = answer.strip('.')
        return answer
    return "UNKNOWN"

async def run_path(path_id: int, prompt: str, temperature: float):
    response = await gemini_generate(prompt, temperature)
    extracted = extract_answer(response)
    return {"path_id": path_id, "text": response, "extracted": extracted}

async def self_consistency_cot(prompt: str, n_paths: int = 5, temperature: float = 0.7):
    print(f"\n==========================================")
    print(f"Executing with N={n_paths}, Temp={temperature}")
    print(f"==========================================\n")

    # Fire tasks in parallel
    tasks = [run_path(i, prompt, temperature) for i in range(n_paths)]
    results = await asyncio.gather(*tasks)

    extracted_answers = []
    html_str = "<div style='display: flex; gap: 20px; overflow-x: auto; font-family: sans-serif;'>"

    for r in results:
        extracted_answers.append(r["extracted"])
        path_text = r['text'].replace('\n', '<br>')
        html_str += f"<div style='flex: 1; border: 1px solid #ccc; padding: 10px; min-width: 300px; background-color: #f9f9f9;'>"
        html_str += f"<b style='color: #333;'>Path {r['path_id']}</b><hr/>"
        html_str += f"<div style='font-size: 0.85em;'>{path_text}</div><hr/>"
        html_str += f"<b style='color: blue;'>Extracted Answer: {r['extracted']}</b>"
        html_str += "</div>"

    html_str += "</div>"
    display(HTML(html_str))

    # Mode Consensus Logic
    valid_answers = [ans for ans in extracted_answers if ans != "UNKNOWN" and "Error" not in ans]

    if not valid_answers:
        consensus = "No valid consensus found"
        confidence = f"0/{n_paths}"
    else:
        counter = Counter(valid_answers)
        best_ans, count = counter.most_common(1)[0]
        consensus = best_ans
        confidence = f"{count}/{n_paths}"

    # Explicitly display the final array and consensus
    display(HTML(f"<div style='margin-top: 20px; padding: 10px; border: 2px solid #4CAF50;'>"
                 f"<h3 style='color: #2E7D32; margin: 0;'>Extracted Mode Consensus Array: {extracted_answers}</h3>"
                 f"<h3 style='color: #C62828; margin: 5px 0 0 0;'>Final Consensus Value: {consensus} (Confidence: {confidence})</h3>"
                 f"</div>"))

    return consensus

## Phase 1: The Classical Baseline Benchmark (T5)
Before testing Prompting architectures, let's observe how older Sequence-to-Sequence models (without inherent step-by-step logic trace generation capabilities) aggressively truncate reasoning and hallucinate zero-shot queries.


In [50]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("Loading FLAN-T5-Small model and tokenizer...")
tokenizer_t5 = AutoTokenizer.from_pretrained("google/flan-t5-small")
model_t5 = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

def t5_pipeline_replacement(prompt_text, max_length=200): # Increased max_length
    input_ids = tokenizer_t5(prompt_text, return_tensors="pt").input_ids
    outputs = model_t5.generate(input_ids, max_length=max_length)
    generated_text = tokenizer_t5.decode(outputs[0], skip_special_tokens=True)
    return [{'generated_text': generated_text}]

prompt_snail = "A snail falls down a 10 foot well. Each day it climbs up 3 feet, but each night it slides down 2 feet. How many days does it take the snail to reach the top? Final Answer: [Your Answer Here]"
print("T5 Prompt:", prompt_snail)

# T5 zero-shot response
t5_response = t5_pipeline_replacement(prompt_snail, max_length=500) # Increased max_length

print("\nClassical Baseline Result:", t5_response[0]['generated_text'])
print("Expected mathematical truth: 8. T5 fails due to inability to reason step-by-step over lengths.")

Loading FLAN-T5-Small model and tokenizer...


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


T5 Prompt: A snail falls down a 10 foot well. Each day it climbs up 3 feet, but each night it slides down 2 feet. How many days does it take the snail to reach the top? Final Answer: [Your Answer Here]

Classical Baseline Result: Each day it climbs 3 feet, so it climbs 3 * 3 = 10 feet. Each day it climbs 2 feet, so it climbs 2 * 10 = 30 feet. It takes the snail 30 + 30 = 90 days to reach the top.
Expected mathematical truth: 8. T5 fails due to inability to reason step-by-step over lengths.


## Phase 1.1: T5 with Chain-of-Thought Prompting
Let's try to improve T5's performance by providing a Chain-of-Thought example within the prompt itself. This is a common technique to guide models to better reasoning.

## Phase 1.2: Encoder-Only Comparison (BERT)
To highlight the difference between 'Understanding' models (Encoders) and 'Reasoning/Generative' models (Seq2Seq/LRMs), let's see how BERT handles the same prompt.

In [42]:
from transformers import pipeline

try:
    # Using a fill-mask pipeline as a proxy for BERT's 'generative' capability
    bert_pipe = pipeline("fill-mask", model="bert-base-uncased")

    print("BERT Test (Snail Problem):")
    # BERT cannot generate long-form answers, it can only predict masked tokens.
    # We provide a lead-in to see what it thinks the 'answer' token should be.
    bert_prompt = "A snail climbs a well. It takes [MASK] days to reach the top."
    bert_results = bert_pipe(bert_prompt)

    for result in bert_results:
        print(f"Predicted token: {result['token_str']} (Score: {result['score']:.4f})")

    print("\nObservation: BERT provides contextual associations but lacks the architecture for multi-step math or logical state tracking.")
except Exception as e:
    print(f"Error loading BERT: {e}")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT Test (Snail Problem):
Predicted token: several (Score: 0.2022)
Predicted token: two (Score: 0.1431)
Predicted token: three (Score: 0.1418)
Predicted token: many (Score: 0.0952)
Predicted token: four (Score: 0.0500)

Observation: BERT provides contextual associations but lacks the architecture for multi-step math or logical state tracking.


## Phase 1.3: BERT with Chain-of-Thought Context
We will now feed the entire reasoning chain into BERT to see if it changes the probability of its top predicted tokens.

In [43]:
bert_cot_prompt = (
    "A snail falls down a 10 foot well. Each day it climbs up 3 feet, but each night it slides down 2 feet. "
    "After 7 days, it is at 7 feet. On day 8, it climbs 3 feet and reaches the top. "
    "Therefore, it takes exactly [MASK] days to reach the top."
)

print("BERT CoT Test:")
bert_cot_results = bert_pipe(bert_cot_prompt)

for result in bert_cot_results:
    print(f"Predicted token: {result['token_str']} (Score: {result['score']:.4f})")

print("\nObservation: Even with the answer provided in the context, BERT picks tokens based on linguistic frequency and local patterns rather than performing a logical 'check' of the math.")

BERT CoT Test:
Predicted token: 10 (Score: 0.1164)
Predicted token: 7 (Score: 0.1041)
Predicted token: 3 (Score: 0.0928)
Predicted token: 4 (Score: 0.0740)
Predicted token: 8 (Score: 0.0723)

Observation: Even with the answer provided in the context, BERT picks tokens based on linguistic frequency and local patterns rather than performing a logical 'check' of the math.


In [27]:
prompt_snail_cot = (
    "Q: A snail falls down a 10 foot well. Each day it climbs up 3 feet, but each night it slides down 2 feet. How many days does it take the snail to reach the top?"
    "A: Let's break this down step by step."
    "On Day 1, the snail climbs 3 feet and slides down 2 feet, so it makes a net progress of 1 foot. It is at 1 foot."
    "On Day 2, it climbs 3 feet (total 4 feet) and slides down 2 feet (total 2 feet). It is at 2 feet."
    "... This pattern continues. For the first 7 days, the snail makes a net progress of 1 foot per day. So after 7 days, it will be at a height of 7 feet."
    "On Day 8, the snail climbs 3 feet. It starts at 7 feet, climbs 3 feet, reaching 10 feet. At this point, it has reached the top of the well and does not slide back down. So it reaches the top on Day 8."
    "Final Answer: 8"
    "\nQ: A snail falls down a 10 foot well. Each day it climbs up 3 feet, but each night it slides down 2 feet. How many days does it take the snail to reach the top? Final Answer: [Your Answer Here]"
)

print("T5 CoT Prompt:", prompt_snail_cot)

# T5 CoT response
t5_cot_response = t5_pipeline_replacement(prompt_snail_cot, max_length=500) # Increased max_length

print("\nClassical Baseline Result with CoT:", t5_cot_response[0]['generated_text'])
print("Expected mathematical truth: 8.")

T5 CoT Prompt: Q: A snail falls down a 10 foot well. Each day it climbs up 3 feet, but each night it slides down 2 feet. How many days does it take the snail to reach the top?A: Let's break this down step by step.On Day 1, the snail climbs 3 feet and slides down 2 feet, so it makes a net progress of 1 foot. It is at 1 foot.On Day 2, it climbs 3 feet (total 4 feet) and slides down 2 feet (total 2 feet). It is at 2 feet.... This pattern continues. For the first 7 days, the snail makes a net progress of 1 foot per day. So after 7 days, it will be at a height of 7 feet.On Day 8, the snail climbs 3 feet. It starts at 7 feet, climbs 3 feet, reaching 10 feet. At this point, it has reached the top of the well and does not slide back down. So it reaches the top on Day 8.Final Answer: 8
Q: A snail falls down a 10 foot well. Each day it climbs up 3 feet, but each night it slides down 2 feet. How many days does it take the snail to reach the top? Final Answer: [Your Answer Here]

Classical Baselin

## Phase 2: The Greedy Baseline Failure (Gemini Deterministic)
Now we push the identical prompt against the enormous Large Reasoning Model (Gemini), but restrict it to `N=1` and `Temperature=0.0`. Notice how single-pass inference is a single point of failure.


In [31]:
# Expectation: Output '10', completely missing the logical boundary condition on Day 8 in a single sweep
await self_consistency_cot(prompt_snail, n_paths=1, temperature=0.0)



Executing with N=1, Temp=0.0



'8 days'

## Phase 3: Self-Consistency Fix (N=5)
We introduce noise via `Temperature=0.7`, running 5 parallel paths. Some wildly fail, some do math wrong, and some find the logical breakpoint. The polling mode correctly pulls out 8!


In [32]:
# Look at the 'Mode Consensus Array' to see the hallucinations vs correct answers voted side-by-side
await self_consistency_cot(prompt_snail, n_paths=5, temperature=0.7)



Executing with N=5, Temp=0.7



'8 days'

## Phase 4: GSM8K Math Problem Proof
Scaling this to massive multidimensional multi-hop arithmetic word problems from the GSM8K dataset.


In [36]:
prompt_gsm = "James buys 5 packs of beef that are 4 pounds each. He also buys 3 packs of chicken that are 2 pounds each. If beef costs $5 a pound and chicken costs $3 a pound, how much does he pay total? Final Answer: [Your Answer Here]"

# T5 test
t5_gsm = t5_pipeline_replacement(prompt_gsm, max_length=50)
print(f"\nT5 Answer to GSM8K: {t5_gsm[0]['generated_text']}\n")

print("\n=========================================")
print("GSM8K with Gemini (N=1, Temp=0.0) - Greedy Baseline")
print("=========================================\n")
await self_consistency_cot(prompt_gsm, n_paths=1, temperature=0.0)

print("\n=========================================")
print("GSM8K with Gemini (N=5, Temp=0.7) - Self-Consistency")
print("=========================================\n")
await self_consistency_cot(prompt_gsm, n_paths=5, temperature=0.7)


T5 Answer to GSM8K: He buys 5 packs of beef and 3 packs of chicken and 3 packs of chicken and 3 packs of chicken and 3 packs of chicken and 3 packs of chicken and 3 packs of chicken and 3 packs of chicken and 3 packs of chicken and 3


GSM8K with Gemini (N=1, Temp=0.0) - Greedy Baseline


Executing with N=1, Temp=0.0




GSM8K with Gemini (N=5, Temp=0.7) - Self-Consistency


Executing with N=5, Temp=0.7



'$118'

## Phase 5: High-Complexity Logic & Optimization Challenge
This challenge involves multiple agents, varying rates, and a scheduling constraint. It is designed to trigger 'step-errors' that Self-Consistency is meant to solve.

In [46]:
prompt_complex = (
    "Problem: A construction crew has 3 workers: Alice, Bob, and Charlie. "
    "1. Alice can build 2 walls per day. Bob can build 1.5 walls per day. Charlie can build 1 wall per day. "
    "2. They need to build 20 walls total. "
    "3. On Day 3, Alice takes a half-day off (works only 0.5 days). "
    "4. On Day 5, the construction site is closed for rain, so no one works. "
    "5. Bob only works every other day, starting on Day 1 (Day 1, 3, 5, etc.). "
    "Question: How many full days (rounding up to the nearest whole day) will it take to complete all 20 walls? "
    "Think step-by-step and track the cumulative walls at the end of every single day until you hit 20. "
    "Final Answer: [Your Answer]"
)

print("\n=========================================")
print("COMPLEX LOGIC: Gemini (N=1, Temp=0.0) - Greedy")
print("=========================================\n")
await self_consistency_cot(prompt_complex, n_paths=1, temperature=0.0)

print("\n=========================================")
print("COMPLEX LOGIC: Gemini (N=5, Temp=0.7) - Self-Consistency")
print("=========================================\n")
await self_consistency_cot(prompt_complex, n_paths=5, temperature=0.7)


COMPLEX LOGIC: Gemini (N=1, Temp=0.0) - Greedy


Executing with N=1, Temp=0.0




COMPLEX LOGIC: Gemini (N=5, Temp=0.7) - Self-Consistency


Executing with N=5, Temp=0.7



'7'

## Phase 6: The Ultimate Constraint Challenge (Extreme Logic)
This problem involves temporal logic, capacity constraints, and interdependent variables. It is designed to be highly sensitive to 'single-pass' errors.

In [51]:
prompt_extreme = (
    "Problem: A high-security elevator has strict rules:\n"
    "1. It starts on Floor 1. Its maximum capacity is 500 lbs.\n"
    "2. There are 4 people: Amy (120 lbs), Ben (200 lbs), Chris (180 lbs), and Dana (150 lbs).\n"
    "3. The elevator takes 10 seconds to move between adjacent floors and 30 seconds for every stop to load/unload people.\n"
    "4. Amy and Chris refuse to be in the elevator together unless Ben is also present.\n"
    "5. Dana must be dropped off at Floor 4. Everyone else must go to Floor 6.\n"
    "Question: What is the minimum time in seconds to get everyone to their required floors starting from the moment the first person steps in at Floor 1? "
    "Provide a step-by-step breakdown of who is in the elevator at each floor and the total time elapsed.\n"
    "Final Answer: [Total Seconds]"
)

print('\n=========================================')
print('EXTREME LOGIC: Gemini (N=1, Temp=0.0) - Greedy')
print('=========================================\n')
await self_consistency_cot(prompt_extreme, n_paths=1, temperature=0.0)

print('\n=========================================')
print('EXTREME LOGIC: Gemini (N=5, Temp=0.7) - Self-Consistency')
print('=========================================\n')
await self_consistency_cot(prompt_extreme, n_paths=7, temperature=0.7)


EXTREME LOGIC: Gemini (N=1, Temp=0.0) - Greedy


Executing with N=1, Temp=0.0




EXTREME LOGIC: Gemini (N=5, Temp=0.7) - Self-Consistency


Executing with N=7, Temp=0.7



'The minimum time in seconds is 110'